# Twin pilot: Llama 3.1 8B on the training window's web interviews

Notebooks 03 and 04 set the supervised baselines. This notebook documents the first run of the language-model twins:
Llama 3.1 8B Instruct predicts the answers of the 2,257 web interviews of April to July 2024, the last four months of
the training window. The pilot has two purposes: to show that the pipeline works end to end, and to check that the
twin's answers are reasonable before it is run on the evaluation window under a preregistered plan. It uses no
evaluation interviews, so that test stays clean.

It covers, in order: how an interview becomes a prompt, how a prompt becomes answer probabilities, how the twin is
calibrated to the survey, the run, and the results against eight checks fixed before the run.

Results are read from `experiments/mvp/2026-10-30-twin-pilot-training-web/`, the GPU run (`python -m sdt.mvp.twin_run
twin-pilot-training-web --stages base,counterfactual,rerun,bf16 --batch-size 16`), and from
`experiments/mvp/2026-10-30-twin-pilot-calibration/`, which redoes the calibration from that run's saved probabilities
without a GPU. Their per-interview files are not in git. Section 2 also runs a small model, Llama 3.2 1B, as an
illustration.

In [1]:
import json, logging, re, warnings
warnings.filterwarnings('ignore', message='IProgress not found')        # quiet library warnings and progress bars
logging.getLogger('torch.distributed.elastic.multiprocessing.redirects').setLevel(logging.ERROR)
import numpy as np, pandas as pd, torch, transformers
transformers.logging.set_verbosity_error(); transformers.logging.disable_progress_bar()
from sdt.mvp import twin
from sdt.mvp.clean import CLASSES, INPUTS, OUT_PATH, OUTCOMES
from sdt.mvp.prompts import FEATURE_SETS, messages
from sdt.mvp.run import EXPERIMENTS
from sdt.mvp.score import log_loss
from sdt.mvp.twin_run import dur_mid_share, interview_sets
pd.set_option('display.max_columns', None); pd.set_option('display.width', None)
ITEMS = list(OUTCOMES)
SHORT = {'demographics': 'D', 'demographics_macro': 'D+M', 'demographics_politics': 'D+P', 'demographics_macro_politics': 'D+M+P'}
DMP = 'demographics_macro_politics'

clean = pd.read_parquet(OUT_PATH)
key = clean.set_index('CASEID')
pilot = interview_sets(clean, ['train-web'])['train-web']
RUN, CAL = EXPERIMENTS / '2026-10-30-twin-pilot-training-web', EXPERIMENTS / '2026-10-30-twin-pilot-calibration'
raw = pd.read_parquet(RUN / 'option_probs.parquet')
pred = pd.read_parquet(CAL / 'predictions.parquet')
calibration = json.load(open(CAL / 'calibration.json'))
cv = pd.read_csv(CAL / 'calibration_cv.csv')
meta = json.load(open(RUN / 'run.json'))
print(f"GPU run of {meta['date']} at commit {meta['commit'][:7]}, {meta['model']} at revision {meta['revision'][:7]}, "
      f"{len(pilot):,} interviews")

def by_cell(s):
    """A Series indexed by (feature set, item) as a table: items down, feature sets across."""
    return s.unstack(0).rename(columns=SHORT)[list(SHORT.values())].loc[ITEMS]

wmean = lambda x, w: np.average(x, weights=w)

GPU run of 2026-10-30 at commit 68b6f47, meta-llama/Llama-3.1-8B-Instruct at revision 0e9e39f, 2,257 interviews


## 1. From interview to prompt

Each prompt asks one item about one interview under one feature set, so the pilot has 2,257 × 5 items × 4 feature sets
= 45,140 prompts. A prompt is two chat messages. The system message introduces the survey, gives the feature set's
state variables and describes the respondent; the user message asks the question. The wording follows five rules:

1. **Questions and answer options are verbatim from the 2024 web questionnaire,** because every evaluation interview
   is web. The options are numbered 1 to 3 in the form's order, in place of Michigan's codes 1, 3 and 5. `DUR` has two
   options, because the web form shows no middle answer.
2. **The introduction keeps the questionnaire's definition of family but not the survey's name,** which could prompt
   the model to recall Michigan's published results.
3. **Each statistic is given now and a year earlier,** the same information as the supervised models' level and change,
   and the parties in office appear without names.
4. **A missing attribute is left out,** because a zero-shot twin cannot learn what "not reported" signals.
5. **No prompt contains a date or a name,** which check 3 in section 5 verifies.

The full text the model receives for one pilot interview, drawn at random among those with every attribute reported,
under D+M+P and item `BUS12`:

In [2]:
ex = pilot[pilot[INPUTS].notna().all(axis=1)].sample(1, random_state=0)
print(f"Interview {ex.CASEID.iloc[0]}, survey month {ex.YYYYMM.iloc[0]}, answer to BUS12: {ex.BUS12.iloc[0]}\n")
text = twin.render(messages(ex, DMP, 'BUS12').iloc[0])
print(text)

Interview 320738, survey month 202407, answer to BUS12: unf

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an adult in the United States taking part in an online survey about the economy.

The questions in the survey will ask you about your family. By family, we mean individuals with whom you combine your incomes and live together in the same household. If you do not combine incomes, or do not live together as a family, please answer these questions thinking only about your own situation.

The latest official statistics:
- Inflation (the change in consumer prices over the past 12 months): 3.3%, compared with 4.0% a year earlier
- Unemployment rate: 4.0%, compared with 3.7% a year earlier
- Economic growth (the change in real GDP over the past four quarters): 2.9%, compared with 1.7% a year earlier
- Federal funds interest rate: 5.33%, compared with 5.07% a year earlier

Politics:
- Party of the president: Democratic
- Party that won the most recent presidential e

The text is in Llama's chat format. `<|start_header_id|>` and `<|end_header_id|>` mark who speaks, and `<|eot_id|>`
ends each turn. The text ends with the header of the assistant's turn, so the next token the model expects is the
start of its reply. Meta's chat template also writes a "Cutting Knowledge Date" line and a "Today Date" line into the
system message. The twin's prompts leave both out, and a test checks that they otherwise match the template character
for character.

## 2. From prompt to probabilities

A language model reads text as tokens: pieces from a fixed vocabulary, 128,256 of them for Llama 3, such as a word,
part of a word or a digit. Given the tokens so far, one forward pass of the model gives a score, called a logit, to
every token in the vocabulary as the next one, and the softmax turns the scores into probabilities. A chatbot writes a
reply by repeating this one token at a time. The twin needs only the first step: the probability that the reply starts
with "1", "2" or "3".

Llama 3.1 8B needs a GPU, so this example runs the same prompt through Llama 3.2 1B, which uses the same tokenizer and
chat format and fits on a laptop. Its numbers illustrate the method; they are not results. The ten most likely first
tokens of its reply:

In [3]:
tok, small = twin.load('meta-llama/Llama-3.2-1B-Instruct')
ids = tok(text, add_special_tokens=False, return_tensors='pt').to(small.device)
with torch.inference_mode():
    p = small(**ids).logits[0, -1].float().softmax(-1)
top = p.topk(10)
print(f'{ids.input_ids.shape[1]} tokens in the prompt')
pd.DataFrame({'token': [repr(tok.decode(i)) for i in top.indices], 'probability': top.values.cpu().numpy().round(4)},
             index=pd.RangeIndex(1, 11, name='rank'))

407 tokens in the prompt


,token,probability
rank,,
1,'3',0.4505
2,'1',0.3646
3,'2',0.1821
4,'4',0.0006
5,'I',0.0005
6,'Part',0.0002
7,'5',0.0002
8,'Good',0.0001
9,'12',0.0001


Nearly all the probability falls on the three option numbers; the rest goes to tokens such as "4" and "I". The twin
turns the distribution into answer probabilities in four steps:

1. Read the probabilities of the tokens "1", "2" and "3".
2. Add them up. The sum is the answer mass: the probability that the reply starts with a valid option.
3. Divide each option's probability by the mass, so the three sum to 1.
4. Map options to classes: 1 to `fav`, 2 to `mid` and 3 to `unf`. For `DUR`, whose form has no middle answer, the
   middle class gets the baseline shares' middle share, 1.1%, and the two options share the rest in their own ratio.

For the interview above, Llama 3.1 8B gave:

In [4]:
r = raw[(raw.stage == 'base') & (raw.feature_set == DMP) & (raw.item == 'BUS12') & (raw.CASEID == ex.CASEID.iloc[0])]
pd.DataFrame({'probability of "1", "2", "3"': r[twin.RAW].iloc[0].to_numpy(),
              'divided by the mass': twin.class_probs(r, 'BUS12')[CLASSES].iloc[0].to_numpy()},
             index=pd.Index(CLASSES, name='class')).T.assign(mass=[r[twin.RAW].sum(axis=1).iloc[0], 1]).round(4)

class,fav,mid,unf,mass
"probability of ""1"", ""2"", ""3""",0.0897,0.7160,0.1935,0.9992
divided by the mass,0.0898,0.7165,0.1936,1.0000


Its median answer mass across the pilot, by item and feature set:

In [5]:
base = raw[raw.stage == 'base']
by_cell(base.assign(mass=base[twin.RAW].sum(axis=1)).groupby(['feature_set', 'item']).mass.median()).round(4)

feature_set,D,D+M,D+P,D+M+P
item,,,,
PAGO,0.9999,0.9999,0.9999,0.9999
PEXP,0.9998,0.9998,0.9999,0.9998
BUS12,0.9997,0.9983,0.9997,0.9989
BUS5,0.9999,0.9999,0.9999,0.9999
DUR,0.9998,0.9998,0.9998,0.9998


The median mass is at least 99.8% in every cell, so the model almost always answers with an option, and dividing by
the mass changes its probabilities very little.

### 2.1 Batches and precision

The GPU runs prompts in batches of 16. Prompts in a batch must have equal length, so shorter ones are padded on the
left with a token that never occurs in text, and an attention mask stops every real token from reading the padding.
Each token's position is counted from its prompt's first real token, so the padding does not shift the positions the
model sees either. A prompt should then get the same probabilities alone as in a batch.

Precision decides whether it does. Meta released the weights in bf16, a 16-bit format that keeps about three
significant digits. Arithmetic in bf16 rounds at every step, and how it rounds depends on the shape of the batch. The run
therefore converts the weights to 32-bit fp32, which is exact, and computes in fp32. Two stages of the run measure both
choices on 200 prompts, 10 per item and feature set, each run alone:

In [6]:
def gap(a, b):
    m = raw[raw.stage == a].merge(raw[raw.stage == b], on=['feature_set', 'item', 'CASEID'])
    d = np.abs(m[[f'{c}_x' for c in twin.RAW]].to_numpy() - m[[f'{c}_y' for c in twin.RAW]].to_numpy())
    return {'prompts': len(m), 'largest difference': np.nanmax(d), 'median difference': np.nanmedian(d)}

pd.DataFrame({'fp32 alone vs fp32 in batches of 16': gap('rerun', 'base'),
              'bf16 alone vs fp32 alone': gap('bf16', 'rerun')}).T.astype({'prompts': int}).style.format(
    {'largest difference': '{:.6f}', 'median difference': '{:.6f}'})

,prompts,largest difference,median difference
fp32 alone vs fp32 in batches of 16,200,0.000007,0.000001
bf16 alone vs fp32 alone,200,0.040489,0.004927


In fp32, batching changes no probability by more than 0.00001. In bf16, the same prompts move by up to 0.04, as much
as many of the differences between models that the study measures.

## 3. Calibration

The twin cannot know two properties of this survey: the base rate of each answer on the web form, and how strongly
differences between people show up in single answers, 47% to 67% of whose variance is transient (notebook 01).
Both describe the instrument rather than the world, so the twin learns them from labelled interviews, while the
direction and relative size of its differences between respondents and states stay the model's.

The calibrated probabilities are softmax(a × log p + b), where p holds the twin's probabilities and b = (b_fav, 0,
b_unf):

- **b, the shift,** moves each answer's log-probability by a constant, and so sets the levels.
- **a, the scale,** multiplies every difference in log-probability: a < 1 shrinks the differences between respondents,
  and a > 1 widens them. a stays at 0 or above, so it can damp the model's structure but not reverse it.

This is bias-corrected temperature scaling (Alexandari, Kundaje and Shrikumar, 2020), which adds the bias b to the
temperature scaling of Guo et al. (2017). a and b are the same for every respondent and are fitted for each feature
set and item by weighted maximum likelihood on the 2,257 pilot interviews. That gives three twins: the zero-shot twin,
uncalibrated; the shifted twin, with b and a = 1; and the scaled twin, with a and b. A scale without a shift is not
used, because with the levels left wrong, a would also absorb part of the level error.

The fitted scale a:

In [7]:
by_cell(pd.Series({(fs, item): c['a'] for fs, d in calibration['scale'].items() for item, c in d.items()})).round(2)

,D,D+M,D+P,D+M+P
PAGO,0.20,0.34,0.20,0.46
PEXP,0.11,0.21,0.14,0.30
BUS12,0.21,0.25,0.39,0.61
BUS5,0.14,0.15,0.23,0.31
DUR,0.39,0.67,0.48,0.95


a is below 1 in every cell and below 0.5 in 17 of 20, so the zero-shot twin's differences between respondents are
more than twice too strong in log-probability in most cells, and up to nine times. It is least overconfident with D+M+P,
on every item.

For the interview of section 1, the three twins give:

In [8]:
row = pred[(pred.CASEID == ex.CASEID.iloc[0]) & (pred.item == 'BUS12') & pred.model.str.endswith(DMP)]
row.set_index(row.model.map({f'twin_{DMP}': 'zero-shot', f'twin_shift_{DMP}': 'shifted', f'twin_scale_{DMP}': 'scaled'}))[
    CLASSES].rename_axis('twin').round(3)

,fav,mid,unf
twin,,,
zero-shot,0.090,0.717,0.194
shifted,0.119,0.523,0.358
scaled,0.140,0.506,0.354


Fitting and scoring on the same interviews would overstate what calibration gains, so a held-out check scores it on
months it was not fitted on:

1. Hold out one of the four months.
2. Fit the shift, the scale and the baseline shares on the other three months, including `DUR`'s middle share.
3. Predict the held-out month.
4. Repeat for each month, pool the four held-out months and compute the weighted log loss.

Skill against the baseline shares on held-out months, in percent:

In [9]:
t = cv.pivot_table(index=['feature_set', 'item'], columns='model', values='log_loss')
skill = pd.concat({name: by_cell(100 * (1 - t[m] / t.baseline)) for m, name in
                   [('twin', 'zero-shot'), ('shift', 'shifted'), ('scale', 'scaled')]}, axis=1)
skill.round(1)

zero-shot                   shifted                   scaled       \
feature_set         D   D+M   D+P D+M+P       D   D+M   D+P D+M+P      D  D+M   
item                                                                            
PAGO            -34.1 -23.0 -36.1 -17.7   -27.2  -5.7 -29.8  -0.7    2.8  2.8   
PEXP            -38.6 -19.3 -36.7 -19.1   -30.1  -8.3 -27.4  -5.2    0.5  0.5   
BUS12           -13.4 -10.2  -9.4  -6.5    -5.5  -3.2  -1.5   1.2    0.3  0.2   
BUS5            -37.1 -47.6 -26.2 -35.9   -16.9 -10.3  -9.3  -4.6    0.5  0.4   
DUR             -11.2  -7.6 -11.1  -8.5    -1.0   0.5  -0.1   1.1    0.6  0.5   

                        
feature_set  D+P D+M+P  
item                    
PAGO         3.7   4.5  
PEXP         1.0   1.3  
BUS12        1.0   1.9  
BUS5         1.0   1.4  
DUR          0.8   1.0

Only the scaled twin beats the baseline shares, in every cell, by 0.2% to 4.5%. The shift alone wins in 3 of 20 cells,
because a twin too sure of its differences between respondents loses more on the respondents it gets wrong than it
gains on the others.

The scale has a cost for the study's question. a is fitted in four months in which the state variables barely move,
so it measures only how strongly the twin's predictions should differ across respondents. It then shrinks the twin's
response to a change of state by the same factor, an amount these months cannot measure. Section 6 returns to this.

## 4. The run

The run used one NVIDIA RTX PRO 6000 Blackwell GPU with 96 GB of memory, rented on RunPod at $2.50 an hour. A benchmark
on 512 of the longest prompts ran at 9.4 to 9.6 prompts per second at every batch size from 16 to 128, so the GPU was
fully used at 16, the size that leaves the most memory free. Four stages ran in order:

1. **Base:** every interview, item and feature set.
2. **Counterfactual:** every interview and item under D+M+P, with the state changed in one of two ways: a Republican
   president and election winner, or inflation of 9.1% with the year-earlier value unchanged.
3. **Rerun:** 200 base prompts, 10 per item and feature set, each run alone.
4. **bf16:** the same 200 prompts, run alone with the weights in bf16.

The run saved its probabilities in chunks, one per stage, scenario, feature set and item, so an interrupted run could
continue from its last finished chunk. It took 1 hour 43 minutes, about $4.30 of GPU time:

In [10]:
pd.DataFrame(meta['per_stage']).T.loc[['base', 'counterfactual', 'rerun', 'bf16']].astype({'prompts': int, 'seconds': int})

,prompts,seconds,prompts_per_second
base,45140,3857,11.70
counterfactual,22570,2293,9.84
rerun,200,23,8.73
bf16,200,8,25.71


At the base stage's 11.7 prompts per second, the preregistered run's roughly 465,000 prompts would take about 11 hours
and cost about $28.

## 5. Results against the eight checks

The eight checks were fixed before the run. Checks 1 to 4 ask whether the pipeline works, and checks 5 to 8 whether
the twin's answers are reasonable:

In [11]:
zs = pred[pred.model.str.match(r'twin_(?!shift|scale)')].assign(fs=lambda d: d.model.str.removeprefix('twin_'))
zs = zs.join(key[['WT', 'YYYYMM', 'POLAFF', 'YTL5'] + ITEMS], on='CASEID')
fav = lambda g: wmean(g.fav, g.WT)
gap_party = zs.groupby(['fs', 'item']).apply(lambda g: fav(g[g.POLAFF == 'Dem']) - fav(g[g.POLAFF == 'Rep']), include_groups=False)
gap_income = zs[zs.item == 'PAGO'].groupby('fs').apply(lambda g: fav(g[g.YTL5 == 5]) - fav(g[g.YTL5 == 1]), include_groups=False)

pattern = re.compile(r'\b(19|20)\d\d\b|January|February|March|April|May|June|July|August|September|October|November|'
                     r'December|Biden|Trump|Harris|Obama|Michigan|Surveys of Consumers|COVID|pandemic|tariff|Powell|Yellen',
                     re.IGNORECASE)
leaks = sum(bool(pattern.search(twin.render(c))) for item in ITEMS for c in messages(pilot, DMP, item))

mid = dur_mid_share(clean)
def scenario_fav(stage, scenario):
    out = {}
    for item, g in raw[(raw.stage == stage) & (raw.scenario == scenario) & (raw.feature_set == DMP)].groupby('item'):
        g = g[key.loc[g.CASEID, item].notna().to_numpy()]
        P = twin.class_probs(g, item, mid).assign(CASEID=g.CASEID.to_numpy()).join(key[['WT', 'POLAFF']], on='CASEID')
        out[item] = {'all': fav(P), 'Rep': fav(P[P.POLAFF == 'Rep']), 'Dem': fav(P[P.POLAFF == 'Dem'])}
    return pd.DataFrame(out).T.loc[ITEMS]
actual_fav = scenario_fav('base', 'actual')
flip, hot = scenario_fav('counterfactual', 'pres_R') - actual_fav, scenario_fav('counterfactual', 'cpi_9.1') - actual_fav

def in_sample(model, item):
    g = pred[(pred.model == model) & (pred.item == item)]
    a = key.loc[g.CASEID]
    return log_loss(a[item].astype(str), g[CLASSES], a.WT)
def shares_loss(item):
    a = pilot[pilot[item].notna()]
    shares = np.array([wmean(a[item] == c, a.WT) for c in CLASSES])
    return log_loss(a[item].astype(str), np.tile(shares, (len(a), 1)), a.WT)
uniform = np.log(3)
beats_uniform = {SHORT[fs]: sum(in_sample(f'twin_{fs}', i) < uniform for i in ITEMS) for fs in FEATURE_SETS}
shift_ok = {SHORT[fs]: sum(in_sample(f'twin_shift_{fs}', i) <= shares_loss(i) for i in ITEMS) for fs in FEATURE_SETS}
sd = zs.groupby(['fs', 'item']).apply(lambda g: np.sqrt(wmean((g.fav - fav(g)) ** 2, g.WT)), include_groups=False)
mass = base[twin.RAW].sum(axis=1).groupby([base.feature_set, base.item]).median()
counts = base.groupby(['feature_set', 'item']).size()

inflation = ', '.join(f"{i} {hot.loc[i, 'all']:+.3f}" for i in ['BUS12', 'BUS5', 'PEXP'])
rows = [
    ('1. Coverage', 'every interview, item and feature set has probabilities summing to 1',
     f'{counts.min():,} interviews in every cell; sums within {np.abs(pred[CLASSES].sum(axis=1) - 1).max():.0e} of 1', 'yes'),
    ('2. Answer mass', 'median at least 0.9 in every cell', f'lowest cell median {mass.min():.4f}', 'yes'),
    ('3. No leaks', 'no prompt contains a year, a month or a listed name', f'{leaks} of {len(pilot) * 5:,} D+M+P prompts', 'yes'),
    ('4. Reproducibility', 'batching changes no probability by more than 0.001', f"largest change {gap('rerun', 'base')['largest difference']:.6f}", 'yes'),
    ('5. Variation', "SD of the twin's fav probability at least 0.02 in every cell", f'smallest {sd.min():.3f}', 'yes'),
    ('6. Known patterns', 'Dem above Rep on all 5 items on D+P and D+M+P; top income quintile above bottom on PAGO',
     f"Dem above Rep on {(gap_party['demographics_politics'] > 0).sum()} items (D+P), {(gap_party[DMP] > 0).sum()} (D+M+P); "
     f'income in {(gap_income > 0).sum()} of 4 feature sets', 'partly'),
    ('7. Counterfactuals', 'Republican president: BUS12 Rep up, Dem down; 9.1% inflation: BUS12, BUS5, PEXP down',
     f"BUS12 Rep {flip.loc['BUS12', 'Rep']:+.3f}, Dem {flip.loc['BUS12', 'Dem']:+.3f}; inflation {inflation}", 'yes'),
    ('8. Accuracy', 'zero-shot beats uniform on 3 of 5 items per feature set; shifted no worse than baseline shares on any item',
     f"items passed (D, D+M, D+P, D+M+P): zero-shot {', '.join(str(n) for n in beats_uniform.values())}; "
     f"shifted {', '.join(str(n) for n in shift_ok.values())}", 'no'),
]
pd.DataFrame(rows, columns=['check', 'passes if', 'result', 'passed']).set_index('check').style.set_properties(**{'text-align': 'left'})

,passes if,result,passed
check,,,
1. Coverage,"every interview, item and feature set has probabilities summing to 1","2,257 interviews in every cell; sums within 2e-16 of 1",yes
2. Answer mass,median at least 0.9 in every cell,lowest cell median 0.9983,yes
3. No leaks,"no prompt contains a year, a month or a listed name","0 of 11,285 D+M+P prompts",yes
4. Reproducibility,batching changes no probability by more than 0.001,largest change 0.000007,yes
5. Variation,SD of the twin's fav probability at least 0.02 in every cell,smallest 0.033,yes
6. Known patterns,Dem above Rep on all 5 items on D+P and D+M+P; top income quintile above bottom on PAGO,"Dem above Rep on 3 items (D+P), 5 (D+M+P); income in 4 of 4 feature sets",partly
7. Counterfactuals,"Republican president: BUS12 Rep up, Dem down; 9.1% inflation: BUS12, BUS5, PEXP down","BUS12 Rep +0.080, Dem -0.062; inflation BUS12 -0.073, BUS5 -0.045, PEXP -0.059",yes
8. Accuracy,zero-shot beats uniform on 3 of 5 items per feature set; shifted no worse than baseline shares on any item,"items passed (D, D+M, D+P, D+M+P): zero-shot 1, 1, 1, 2; shifted 0, 1, 0, 2",no


The pipeline passes all four of its checks. The twin's answers vary across respondents and respond to the state in the
right direction, but it fails on the partisan pattern of D+P and on accuracy. Sections 5.1 to 5.3 explain the three
patterns behind these results.

### 5.1 The middle answer

The zero-shot twin's answer shares under D+M+P against the actual shares, weighted, for the pilot interviews that
answered each item:

In [12]:
d = zs[zs.fs == DMP]
shares = {item: {**{('twin', c): wmean(g[c], g.WT) for c in CLASSES}, **{('actual', c): wmean(g[item] == c, g.WT) for c in CLASSES}}
          for item, g in d.groupby('item')}
pd.DataFrame(shares).T.loc[ITEMS].round(3)

twin               actual              
         fav    mid    unf    fav    mid    unf
PAGO   0.224  0.540  0.236  0.253  0.287  0.461
PEXP   0.134  0.727  0.139  0.282  0.476  0.242
BUS12  0.130  0.675  0.195  0.173  0.488  0.339
BUS5   0.059  0.881  0.060  0.134  0.603  0.263
DUR    0.581  0.011  0.407  0.397  0.011  0.592

On every item that offers a middle answer, the zero-shot twin gives it far more probability than respondents do, so
the twin hedges where people take a side. The shift corrects the levels, and with them the average of this bias.

### 5.2 Differences between respondents

The gap in favourable answers between Democrats and Republicans on each item, and between the top and bottom income
quintiles on `PAGO`, for the zero-shot twin and the actual answers:

In [13]:
d = zs[zs.fs == DMP]
actual_fav_share = lambda g, item: wmean(g[item] == 'fav', g.WT)
actual_party = pd.Series({item: actual_fav_share(g[g.POLAFF == 'Dem'], item) - actual_fav_share(g[g.POLAFF == 'Rep'], item)
                          for item, g in d.groupby('item')})
g = d[d.item == 'PAGO']
actual_income = actual_fav_share(g[g.YTL5 == 5], 'PAGO') - actual_fav_share(g[g.YTL5 == 1], 'PAGO')
t = by_cell(gap_party).assign(actual=actual_party)
t.index = [f'Dem − Rep, {i}' for i in t.index]
t.loc['top − bottom quintile, PAGO'] = [*gap_income.rename(SHORT)[list(SHORT.values())], actual_income]
t.round(3)

fs,D,D+M,D+P,D+M+P,actual
"Dem − Rep, PAGO",-0.081,-0.043,0.018,0.041,0.286
"Dem − Rep, PEXP",-0.070,-0.023,0.064,0.050,0.178
"Dem − Rep, BUS12",-0.062,-0.040,-0.008,0.043,0.260
"Dem − Rep, BUS5",-0.066,-0.063,-0.020,0.017,0.092
"Dem − Rep, DUR",-0.031,0.002,0.007,0.038,0.213
"top − bottom quintile, PAGO",0.680,0.486,0.802,0.436,0.311


The twin's differences between respondents are too strong on income and too weak on party. Its income gap on `PAGO` is
1.4 to 2.6 times the actual gap. Its partisan gaps are a fraction of the actual ones. On D it puts Republicans above
Democrats on every item, the reverse of these months under a Democratic president; on D+P it still reverses two items,
and on D+M+P it gets every direction right. The scale a cannot fix both errors, because it shrinks every difference by
the same factor.

### 5.3 Counterfactuals

The change in the zero-shot twin's mean favourable probability under D+M+P when the state is changed, overall and by
party:

In [14]:
pd.concat({'Republican president': flip, 'inflation 9.1%': hot}, axis=1).round(3)

Republican president               inflation 9.1%              
                       all    Rep    Dem            all    Rep    Dem
PAGO                 0.010  0.102 -0.063         -0.110 -0.122 -0.117
PEXP                 0.004  0.071 -0.050         -0.059 -0.057 -0.069
BUS12                0.000  0.080 -0.062         -0.073 -0.068 -0.094
BUS5                 0.008  0.102 -0.053         -0.045 -0.056 -0.064
DUR                 -0.003  0.030 -0.029         -0.173 -0.188 -0.160

A Republican president raises Republicans' favourable answers and lowers Democrats' on every item, so the twin knows
that partisans judge the economy by who governs. Inflation of 9.1% lowers favourable answers on every item, most of
all on whether it is a good time to buy household goods. The Republican scenario combines the economy of mid-2024 with
a party in office that never went with it, so the twin's response cannot come from recalling a period. The inflation
scenario is weaker evidence: 9.1% was the actual peak of June 2022, a period the model may know.

## 6. What the pilot establishes

- **The pipeline works.** The model answers almost every prompt with an option, batching changes no probability by
  more than 0.00001, and no prompt carries a date or a name. The preregistered run uses the same command, with the evaluation
  interviews added.
- **The zero-shot twin predicts individual answers poorly.** It overuses the middle answer, exaggerates the income
  gradient and understates the partisan gap.
- **The twin responds to the state in the right direction,** including to a combination of president and economy that
  never occurred.
- **Calibration with three numbers per cell makes the twin beat the baseline shares on held-out months, but only by up
  to 4.5%.** Its scale is learned in months in which the state barely moves, and it damps the twin's response to a
  change of state by the same factor as its differences between respondents. So the preregistration has to choose the
  twin that carries the main claim: the shifted twin, whose response to a new state is entirely the model's, or the
  scaled twin, which predicts individual answers better.